### Propósito de este scraper y página objetivo:

- El objetivo es scrapear un archivo en formato excel de Odepa que contiene los datos de los precios de la naranja.

- En este caso, se extrajo la fecha, producto, variedad, calidad y precio proemdio en peso chileno

In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['previa_proyecto']
coleccion = db['naranja']

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [3]:
import pandas as pd
import time
from pymongo import MongoClient
from collections import OrderedDict
import os

# --- CONFIGURACIÓN ---
NOMBRE_GRUPO = "G9-AgroTech"
NOMBRE_INTEGRANTE = "Sebastián Castillo"
NOMBRE_ARCHIVO = "precio-consumidor_semanal_202501-202552.xlsx" 

# --- PASO 1: LECTURA ---
try:
    if NOMBRE_ARCHIVO.endswith('.xlsx'):
        df = pd.read_excel(NOMBRE_ARCHIVO, skiprows=4)
    else:
        df = pd.read_csv(NOMBRE_ARCHIVO, skiprows=4)
    
    # Seleccionamos las columnas originales del Excel
    columnas_originales = ['Fecha inicio', 'Producto', 'Variedad', 'Calidad', 'Precio promedio']
    df_final = df[columnas_originales].copy()
    
    # Limpiamos filas vacías
    df_final = df_final.dropna(subset=['Producto'])
    print(f"✅ Archivo cargado correctamente: {len(df_final)} registros.")

except Exception as e:
    print(f"❌ Error al procesar el archivo: {e}")
    df_final = None

# --- PASO 2: CARGA A MONGODB CON RENOMBRADO ---
if df_final is not None:
    try:
        client = MongoClient('mongodb://bigdata_mongodb:27017/')
        db = client['previa_proyecto']
        coleccion = db['naranja']
        
        registros = df_final.to_dict(orient='records')
        exitos = 0
        
        print("--- Iniciando carga en colección 'naranja' ---")
        
        for fila in registros:
            try:
                # Usamos OrderedDict para controlar el orden de los campos
                doc = OrderedDict()
                doc["grupo"] = NOMBRE_GRUPO
                doc["Integrante"] = NOMBRE_INTEGRANTE
                
                # RENOMBRADO AQUÍ: De 'Fecha inicio' (Excel) a 'fecha' (Mongo)
                doc["Fecha"] = fila["Fecha inicio"]
                
                doc["Producto"] = fila["Producto"]
                doc["Variedad"] = fila["Variedad"]
                doc["Calidad"] = fila["Calidad"]
                
                # Limpieza de precio
                precio_raw = str(fila["Precio promedio"]).replace('.', '').replace(',', '.')
                doc["Precio promedio"] = float(precio_raw)
                
                # Columna de la extrema derecha
                doc["Fecha de captura"] = time.strftime("%Y-%m-%d %H:%M:%S")
                
                coleccion.insert_one(doc)
                exitos += 1
            except:
                continue
        
        print(f"✅ CARGA EXITOSA: {exitos} registros guardados.")
        client.close()
        
    except Exception as e:
        print(f"❌ Error de conexión: {e}")

✅ Archivo cargado correctamente: 162 registros.
--- Iniciando carga en colección 'naranja' ---
✅ CARGA EXITOSA: 162 registros guardados.


/opt/conda/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
